In [ ]:
import pandas as pd
import numpy as np
import optuna
import warnings
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate

# Keep terminal clean
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

def apply_feature_engineering(df, params):
    """Rebuilds the dataset exactly how Optuna mutated it."""
    X_mutated = df.copy()
    base_cols = list(df.columns)
    
    for i in range(3):
        if params.get(f"build_interaction_{i}", False):
            c1, c2, op = params[f"inter_{i}_c1"], params[f"inter_{i}_c2"], params[f"inter_{i}_op"]
            col_name = f"{c1}_{op}_{c2}"
            if op == "add": X_mutated[col_name] = X_mutated[c1] + X_mutated[c2]
            elif op == "sub": X_mutated[col_name] = X_mutated[c1] - X_mutated[c2]
            elif op == "mult": X_mutated[col_name] = X_mutated[c1] * X_mutated[c2]
            elif op == "div": X_mutated[col_name] = X_mutated[c1] / (X_mutated[c2] + 1e-9)

    for col in base_cols:
        if params.get(f"square_{col}", False):
            X_mutated[f"{col}_squared"] = X_mutated[col] ** 2

    cols_to_drop = [c for c in X_mutated.columns if params.get(f"drop_prob_{c}", 0.0) > 0.8]
    if len(cols_to_drop) < len(X_mutated.columns):
        X_mutated = X_mutated.drop(columns=cols_to_drop)
        
    return X_mutated

if __name__ == "__main__":
    TARGET_STOCK = 1  # <--- SET YOUR STOCK NUMBER HERE
    TARGET_AGGRESSION = 9 
    
    print(f"========== 🚀 INITIATING QUANT PIPELINE: STOCK {TARGET_STOCK} 🚀 ==========")
    
    try:
        train = pd.read_csv(f"hackathon_data/stock_{TARGET_STOCK}_train.csv")
        test = pd.read_csv(f"hackathon_data/stock_{TARGET_STOCK}_test.csv")
    except FileNotFoundError:
        print(f"[!] Data for Stock {TARGET_STOCK} not found. Exiting.")
        exit()

    X_full_raw = train.drop("target", axis=1)
    y_full_raw = train["target"]
    num_rows = len(X_full_raw)
    
    # --- 🛡️ FLASH CRASH DEFENSE ---
    lower_bound = y_full_raw.quantile(0.01)
    upper_bound = y_full_raw.quantile(0.99)
    y_full = y_full_raw.clip(lower=lower_bound, upper=upper_bound)

    def objective(trial):
        # 🧬 1. GENETIC FEATURE MUTATOR
        X_mutated = X_full_raw.copy()
        base_cols = list(X_mutated.columns)
        
        for i in range(3):
            if trial.suggest_categorical(f"build_interaction_{i}", [True, False]):
                c1 = trial.suggest_categorical(f"inter_{i}_c1", base_cols)
                c2 = trial.suggest_categorical(f"inter_{i}_c2", base_cols)
                op = trial.suggest_categorical(f"inter_{i}_op", ["add", "sub", "mult", "div"])
                col_name = f"{c1}_{op}_{c2}"
                if op == "add": X_mutated[col_name] = X_mutated[c1] + X_mutated[c2]
                elif op == "sub": X_mutated[col_name] = X_mutated[c1] - X_mutated[c2]
                elif op == "mult": X_mutated[col_name] = X_mutated[c1] * X_mutated[c2]
                elif op == "div": X_mutated[col_name] = X_mutated[c1] / (X_mutated[c2] + 1e-9)

        for col in base_cols:
            if trial.suggest_categorical(f"square_{col}", [True, False]):
                X_mutated[f"{col}_squared"] = X_mutated[col] ** 2
                
        cols_to_drop = []
        for col in X_mutated.columns:
            if trial.suggest_float(f"drop_prob_{col}", 0.0, 1.0) > 0.8:
                cols_to_drop.append(col)
        if len(cols_to_drop) < len(X_mutated.columns):
            X_mutated = X_mutated.drop(columns=cols_to_drop)

        # 🛡️ 2. DATA SANITIZER (Poison Pill Defense)
        X_mutated = X_mutated.replace([np.inf, -np.inf], np.nan).fillna(0)

        # 🤖 3. MODEL BUILDER
        scaler_str = trial.suggest_categorical("scaler", ["Standard", "Robust"])
        scaler = StandardScaler() if scaler_str == "Standard" else RobustScaler()
        
        if num_rows < 500:
            m_type = trial.suggest_categorical("model", ["Ridge", "Lasso"])
        else:
            m_type = trial.suggest_categorical("model", ["Ridge", "Lasso", "HistGBM", "XGBoost", "LightGBM"])
        
        if m_type == "Ridge":
            model = Ridge(alpha=trial.suggest_float("alpha", 1e-3, 10.0, log=True), random_state=42)
        elif m_type == "Lasso":
            model = Lasso(alpha=trial.suggest_float("alpha", 1e-4, 1.0, log=True), random_state=42, max_iter=2000)
        elif m_type == "XGBoost":
            model = xgb.XGBRegressor(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7),
                subsample=trial.suggest_float("subsample", 0.5, 1.0),
                random_state=42, n_jobs=-1,
                tree_method="hist", device="cuda" # GPU Enabled
            )
        elif m_type == "LightGBM":
            lgb_depth = trial.suggest_int("max_depth", 3, 7)
            max_leaves = (2 ** lgb_depth) - 1
            lgb_leaves = trial.suggest_int("num_leaves", 2, min(40, max_leaves)) # Math Paradox Fix
            model = lgb.LGBMRegressor(
                n_estimators=trial.suggest_int("n_estimators", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=lgb_depth, num_leaves=lgb_leaves,
                random_state=42, n_jobs=-1, verbose=-1
            )
        else:
            model = HistGradientBoostingRegressor(
                max_iter=trial.suggest_int("max_iter", 100, 500),
                learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                max_depth=trial.suggest_int("max_depth", 3, 7), random_state=42
            )

        pipeline = Pipeline([('scaler', scaler), ('model', model)])

        # 📊 4. TIME-SERIES VALIDATION
        tscv = TimeSeriesSplit(n_splits=5)
        scores = cross_validate(pipeline, X_mutated, y_full, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
        return -scores['test_score'].mean()

    # --- RUN OPTUNA ---
    print(f"[Optuna] Searching for Alpha (1000 Trials)...")
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    study.optimize(objective, n_trials=1000)
    
    best_params = study.best_params
    print(f"\n[✔] Tuning Complete! Best MAE: {study.best_value:.4f} | Model: {best_params['model']}")

    # ========================================================
    # 🏦 THE TRADING FLOOR (EXECUTION)
    # ========================================================
    print(f"\n========== 🏦 LIVE EXECUTION ENGINE 🏦 ==========")
    
    # 1. Apply DNA & Sanitize
    X_train = apply_feature_engineering(X_full_raw, best_params).replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test = apply_feature_engineering(test, best_params).replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # 2. Rebuild Final Pipeline
    scaler = StandardScaler() if best_params['scaler'] == "Standard" else RobustScaler()
    if best_params['model'] == "Ridge": final_model = Ridge(alpha=best_params['alpha'], random_state=42)
    elif best_params['model'] == "Lasso": final_model = Lasso(alpha=best_params['alpha'], random_state=42)
    elif best_params['model'] == "HistGBM": final_model = HistGradientBoostingRegressor(max_iter=best_params['max_iter'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], random_state=42)
    elif best_params['model'] == "XGBoost": final_model = xgb.XGBRegressor(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], subsample=best_params['subsample'], random_state=42, n_jobs=-1, tree_method="hist", device="cuda")
    elif best_params['model'] == "LightGBM": final_model = lgb.LGBMRegressor(n_estimators=best_params['n_estimators'], learning_rate=best_params['learning_rate'], max_depth=best_params['max_depth'], num_leaves=best_params['num_leaves'], random_state=42, n_jobs=-1, verbose=-1)
    
    final_pipeline = Pipeline([('scaler', scaler), ('model', final_model)])
    
    # 3. Final Risk Assessment
    cv_results = cross_validate(final_pipeline, X_train, y_full, cv=TimeSeriesSplit(n_splits=5), scoring='neg_mean_absolute_error', n_jobs=-1)
    mean_mae = (-cv_results['test_score']).mean()
    cv_vol = (-cv_results['test_score']).std() / mean_mae
    
    # 4. Dynamic Strategy & Guardrails
    dynamic_multiplier = min(1.0, max(0.05, ((11 - TARGET_AGGRESSION) / 10.0) + (cv_vol * 5.0)))
    spread_margin = np.clip(mean_mae * dynamic_multiplier, a_min=0.50, a_max=15.00)
    
    # 5. Predict & Skew
    final_pipeline.fit(X_train, y_full)
    mid_preds = final_pipeline.predict(X_test)
    historical_mean = y_full.tail(50).mean()
    
    print(f"\n{'ROW':<5} | {'MID (PRED)':<12} | {'BID':<10} | {'ASK':<10} | {'SPREAD':<8}")
    print("-" * 65)
    
    # Print the first 10 rows to the console
    for i in range(min(10, len(mid_preds))):
        momentum_shift = (mid_preds[i] - historical_mean) * 0.10
        skewed_mid = mid_preds[i] + momentum_shift
        final_bid = skewed_mid - spread_margin
        final_ask = skewed_mid + spread_margin
        
        print(f"{i:<5} | {mid_preds[i]:<12.4f} | {final_bid:<10.4f} | {final_ask:<10.4f} | {spread_margin*2:<8.4f}")
        
    print("-" * 65)
    print(f"[✔] Pipeline execution complete for Stock {TARGET_STOCK}. MAE: {mean_mae:.3f} | Volatility: {cv_vol*100:.1f}%")